# 03_子Agent流式 (Subagent Streaming)

**来源:** [LangChain Docs — Subagent Streaming Frontend](https://docs.langchain.com/deep-agents/frontend/subagent-streaming)

当协调器智能体派发专门子智能体（研究员、分析员、写手）时，需要将协调器的消息与每个子智能体的流式输出分开渲染。本教程演示如何使用 `filterSubagentMessages` 和 `getSubagentsByMessage` 构建清晰的子智能体卡片 UI。

## 1. 为什么需要过滤子智能体消息

不进行过滤时，每个子智能体产生的每个 Token 都会交错出现在协调器的消息流中，导致 UI 不可读。

| 特性 | 说明 |
|------|------|
| `filterSubagentMessages: true` | `stream.messages` 只包含协调器消息 |
| `stream.subagents` | 访问所有子智能体的独立流 |
| `stream.getSubagentsByMessage(id)` | 获取指定协调器消息派发的子智能体列表 |

## 2. 设置 useStream

始终设置 `filterSubagentMessages: true`，并定义 TypeScript 接口匹配智能体的状态模式以获得类型安全。

In [ ]:
# React 前端设置示例（TypeScript）：
#
# import type { BaseMessage } from "@langchain/core/messages";
#
# interface AgentState {
#   messages: BaseMessage[];
# }
#
# import { useStream } from "@langchain/react";
#
# const AGENT_URL = "http://localhost:2024";
#
# export function DeepAgentChat() {
#   const stream = useStream<typeof myAgent>({
#     apiUrl: AGENT_URL,
#     assistantId: "deep_agent_subagent_cards",
#     filterSubagentMessages: true,
#   });
#
#   return (
#     <div>
#       {stream.messages.map((msg) => (
#         <MessageWithSubagents
#           key={msg.id}
#           message={msg}
#           subagents={stream.getSubagentsByMessage(msg.id)}
#         />
#       ))}
#     </div>
#   );
# }

print("useStream 配置：filterSubagentMessages: true 分离协调器与子智能体消息")

## 3. 提交消息与子图流式

提交消息时启用子图流式，并设置适当的递归限制。

In [ ]:
# 提交消息时启用子图流式：
#
# stream.submit(
#   { messages: [{ type: "human", content: text }] },
#   { streamSubgraphs: true }
# );
#
# Deep Agents 默认递归限制为 10,000，足够大多数多专家场景

print("streamSubgraphs: true 确保子智能体的流式数据也传输到前端")

## 4. SubagentStreamInterface

每个子智能体暴露一个 `SubagentStreamInterface` 对象，包含元数据：

| 属性 | 类型 | 说明 |
|------|------|------|
| `id` | `string` | 子智能体实例唯一标识 |
| `status` | `pending\|running\|complete\|error` | 生命周期状态 |
| `messages` | `BaseMessage[]` | 子智能体自身的消息流，实时更新 |
| `result` | `string\|undefined` | 最终输出文本，仅 complete 时可访问 |
| `toolCall` | `object` | 派发此子智能体的工具调用信息 |
| `toolCall.args.description` | `string` | 协调器分配给子智能体的任务描述 |
| `toolCall.args.subagent_type` | `string` | 子智能体类型（如 researcher, analyst） |
| `startedAt` | `number\|undefined` | 开始执行的时间戳 |
| `completedAt` | `number\|undefined` | 完成执行的时间戳 |

## 5. 构建 SubagentCard 组件

每个子智能体卡片显示名称、任务描述、流式内容和时间信息。

In [ ]:
# React 子智能体卡片组件（TypeScript）：
#
# import { AIMessage } from "@langchain/core/messages";
#
# function SubagentCard({ subagent }: { subagent: SubagentStreamInterface }) {
#   const [expanded, setExpanded] = useState(true);
#   const title = subagent.toolCall?.args?.subagent_type ?? `Agent ${subagent.id}`;
#   const description = subagent.toolCall?.args?.description ?? "";
#
#   const lastAIMessage = subagent.messages
#     .filter(AIMessage.isInstance)
#     .at(-1);
#
#   const displayContent =
#     subagent.status === "complete"
#       ? subagent.result
#       : typeof lastAIMessage?.content === "string"
#         ? lastAIMessage.content
#         : "";
#
#   return (
#     <div className="rounded-lg border bg-white shadow-sm">
#       <button onClick={() => setExpanded(!expanded)} className="flex w-full items-center justify-between p-4">
#         <div className="flex items-center gap-3">
#           <StatusIcon status={subagent.status} />
#           <div>
#             <h4 className="font-semibold capitalize">{title}</h4>
#             <p className="text-xs text-gray-500">{description}</p>
#           </div>
#         </div>
#         <StatusBadge status={subagent.status} />
#       </button>
#       {expanded && displayContent && (
#         <div className="border-t px-4 py-3">
#           <div className="prose prose-sm max-w-none line-clamp-6">{displayContent}</div>
#         </div>
#       )}
#     </div>
#   );
# }

print("SubagentCard：可折叠卡片，显示状态图标、任务描述和流式内容")

## 6. 状态图标与徽章

| 状态 | 图标 | 徽章样式 |
|------|------|----------|
| pending | `○` 灰色 | `bg-gray-100 text-gray-600` |
| running | `◉` 蓝色旋转 | `bg-blue-100 text-blue-700` |
| complete | `✓` 绿色 | `bg-green-100 text-green-700` |
| error | `✕` 红色 | `bg-red-100 text-red-700` |

## 7. 进度追踪

显示进度条和计数器，让用户了解子智能体的完成情况。

In [ ]:
# React 进度条组件：
#
# function SubagentProgress({ subagents }: { subagents: SubagentStreamInterface[] }) {
#   const completed = subagents.filter((s) => s.status === "complete").length;
#   const total = subagents.length;
#   const percentage = total > 0 ? Math.round((completed / total) * 100) : 0;
#
#   return (
#     <div className="space-y-1">
#       <div className="flex items-center justify-between text-xs text-gray-500">
#         <span>Subagent progress</span>
#         <span>{completed}/{total} complete</span>
#       </div>
#       <div className="h-2 overflow-hidden rounded-full bg-gray-200">
#         <div className="h-full rounded-full bg-blue-500 transition-all duration-300"
#           style={{ width: `${percentage}%` }} />
#       </div>
#     </div>
#   );
# }

print("SubagentProgress：实时显示子智能体完成比例")

## 8. 消息与子智能体卡片布局

核心布局模式：渲染每条协调器消息，如果该消息派生了子智能体，在消息下方立即渲染子智能体卡片。

### 合成指示器

所有子智能体完成后，协调器需要时间合成为最终响应。显示清晰的「正在合成…」指示器，防止用户误以为智能体卡住了。

In [ ]:
# React 合成指示器：
#
# function SynthesisIndicator({ subagents, isLoading }) {
#   const allComplete = subagents.length > 0 &&
#     subagents.every((s) => s.status === "complete" || s.status === "error");
#
#   if (!allComplete || !isLoading) return null;
#
#   return (
#     <div className="flex items-center gap-2 rounded-lg bg-purple-50 px-4 py-2 text-sm text-purple-700">
#       <span className="animate-spin">⟳</span>
#       Synthesizing results from {subagents.length} subagent{subagents.length !== 1 ? "s" : ""}...
#     </div>
#   );
# }

print("合成指示器：所有子智能体完成后显示紫色加载提示")

## 9. 最佳实践

| 实践 | 说明 |
|------|------|
| 始终设置 `filterSubagentMessages: true` | 未过滤的流式数据无法阅读 |
| 显示任务描述 | `toolCall.args.description` 让用户了解子智能体的任务 |
| 使用可折叠卡片 | 5+ 子智能体时自动折叠已完成卡片 |
| 显示时间数据 | 展示每个子智能体的耗时，帮助理解性能和瓶颈 |
| 设置适当的递归限制 | 嵌套子图工作流需要高于默认值 25，建议至少 100 |
| 逐子智能体处理错误 | 一个子智能体失败不应导致整个 UI 崩溃 |

## 10. 使用场景

| 场景 | 说明 |
|------|------|
| 深度研究 | 协调器派发研究员调查问题的不同方面 |
| 多专家分析 | 法律、财务、技术等领域专家各抒己见 |
| 复杂任务分解 | 规划器将大任务拆分为子任务 |
| 代码审查管线 | 安全审查、风格检查、性能分析并行执行 |

---

**参考链接**
- [LangChain Docs — Subagent Streaming](https://docs.langchain.com/deep-agents/frontend/subagent-streaming)
- [useStream API 参考](https://docs.langchain.com/react/use-stream)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)